Import libraries for building learning models and analytical graphs

In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


Build dataframes for train set and test set from csv files

In [ ]:
train = pd.read_csv('cell2celltrain.csv')
test = pd.read_csv('cell2cellholdout.csv')

In [ ]:
train.head()

In [ ]:
test.head()

## Data Analysis and Preprocessing

Since 'CustomerID' and 'ServiceArea' don't really affect the churn result, we can remove these columns from the dataset.

In [ ]:
train.drop(['CustomerID', 'ServiceArea'], axis=1, inplace=True)
test.drop(['CustomerID', 'ServiceArea'], axis=1, inplace=True)

### Churn - NonChurn analysis

In [ ]:
churnIndex = train[train['Churn']=='Yes'].shape[0]
nonChurnIndex = train[train['Churn']=='No'].shape[0]

plt.pie([churnIndex, nonChurnIndex], explode = [0.1, 0], labels = ['Churn','Non Churn'], autopct='%1.1f%%', startangle=0)  
plt.show()

### Categories of dataset

In [ ]:
print("Categorical Var. : {}".format(train.select_dtypes(exclude=['int', 'float']).columns))

print("Numeric Var. : {}".format(train.select_dtypes(include=['int', 'float']).columns))

### Missing values

Detect columns with missing values

In [ ]:
train.columns[train.isna().any()].tolist()

In [ ]:
test.columns[test.isna().any()].tolist()

In [ ]:
train_missings = train.filter(['MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'Handsets',
 'HandsetModels',
 'CurrentEquipmentDays',
 'AgeHH1',
 'AgeHH2',
 'Homeownership',
 'HandsetPrice',
 'MaritalStatus'], axis=1)
test_missings = test.filter(['Churn',
 'MonthlyRevenue',
 'MonthlyMinutes',
 'TotalRecurringCharge',
 'DirectorAssistedCalls',
 'OverageMinutes',
 'RoamingCalls',
 'PercChangeMinutes',
 'PercChangeRevenues',
 'AgeHH1',
 'AgeHH2',
 'Homeownership',
 'HandsetPrice',
 'MaritalStatus'], axis=1)

Illustrates the missing-value rates to eliminate columns with high missing-value rate: 

In [ ]:
print("Missing-value rate of train data:")
mis_val_train = train_missings.isnull().sum() / len(train_missings) * 100
print(mis_val_train)
print()
print("Missing-value rate of test data:")
mis_val_test = test_missings.isnull().sum() / len(test_missings) * 100
print(mis_val_test)

'Homeownership', 'HandsetPrice', 'MaritalStatus' have high rate of missing values. Therefore, we should drop these columns from the datasets.

In [ ]:
train.drop(['Homeownership', 'HandsetPrice', 'MaritalStatus'], axis=1, inplace=True)
test.drop(['Homeownership', 'HandsetPrice', 'MaritalStatus'], axis=1, inplace=True)

### Outliers detection

In [ ]:
print ("There are " + str(train.shape[1]) + " columns and  " + str(train.shape[0]) + " rows in the train data.\n")
print("There are " + str(test.shape[1]) + " columns and " + str(test.shape[0]) + " rows in the test data.\n")

In [ ]:
from scipy.stats import zscore

train_outliers=train.select_dtypes(include=['int', 'float'])
train_outliers.head()

Check number of outliers using "zscore":

In [ ]:
outliers1={}
for col in train_outliers:
    z_score = zscore(train_outliers[col])
    upper_replace_value = train_outliers[z_score<=3][col].max()
    lower_replace_value = train_outliers[z_score>=-3][col].min()
    outliers1[col] = len(train_outliers[col][(train_outliers[col] > upper_replace_value)| (train_outliers[col] < lower_replace_value)])
outliers_zscore = pd.DataFrame(outliers1.items(),columns=['Feature','Outliers'])
print(outliers_zscore)

Replace outliers with upper/lower replace value:

In [ ]:
'''train_new = train_outliers.copy()
outliers2={}
for col in train_outliers:
    z_score = zscore(train_outliers[col])
    upper_replace_value = train_outliers[z_score<=3][col].max()
    lower_replace_value = train_outliers[z_score>=-3][col].min()
    train_new.loc[train_outliers[col] > upper_replace_value, col] = upper_replace_value
    train_new.loc[train_outliers[col] < lower_replace_value, col] = lower_replace_value
    outliers2[col] = len(train_new[col][(train_new[col] > upper_replace_value)| (train_new[col] < lower_replace_value)])
outliers_zscore = pd.DataFrame(outliers2.items(),columns=['Feature','Outliers'])
print(outliers_zscore)'''

outliers2={}
for col in train_outliers:
    z_score = zscore(train_outliers[col])
    upper_replace_value = train_outliers[z_score<=3][col].max()
    lower_replace_value = train_outliers[z_score>=-3][col].min()
    train.loc[train_outliers[col] > upper_replace_value, col] = upper_replace_value
    train.loc[train_outliers[col] < lower_replace_value, col] = lower_replace_value
    outliers2[col] = len(train[col][(train[col] > upper_replace_value)| (train[col] < lower_replace_value)])
outliers_zscore = pd.DataFrame(outliers2.items(),columns=['Feature','Outliers'])
print(outliers_zscore)

### Label encoding

Label encode categorical data

In [ ]:
from sklearn.preprocessing import LabelEncoder

train_df=train.copy()

enc = LabelEncoder()

for col in train_df.columns:
    if train_df.dtypes[col] == object:
        enc.fit(train_df[col].astype(str))
        train_df[col] = enc.transform(train_df[col].astype(str))
train_df.info()

In [ ]:
test_df=test.copy()

for col in test_df.columns:
    if test_df.dtypes[col] == object:
        enc.fit(test_df[col].astype(str))
        test_df[col] = enc.transform(test_df[col].astype(str))
test_df.info()

### Feature scaling & Train-Test split

Divide train set to feature matrix X and target vector y

In [ ]:
y = train_df['Churn']
X= train_df.drop(columns=['Churn'],axis=1)

X.head()

In [ ]:
y.head()

Inside the train set, give 80% of the data to be used for learning and 20% for testing learning models

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=999,stratify=y)

Perform standardization on feature matrix X

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[X_train.columns] = scaler.fit_transform(X_train[X_train.columns])
X_test[X_test.columns] = scaler.transform(X_test[X_test.columns])

## Training model with different algorithms

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')

logReg_X_train = X_train.copy()

logReg_X_train[logReg_X_train.columns] = imputer.fit_transform(logReg_X_train[logReg_X_train.columns])

# Create and fit the logistic regression model
logReg = LogisticRegression()
logReg.fit(logReg_X_train, y_train)

y_pred = logReg.predict(logReg_X_train)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_train, y_pred)
print("Accuracy:", accuracy)

### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Create a mean imputer
imputer = SimpleImputer(strategy='mean')

# Impute missing values in X_train
X_train_imputed = imputer.fit_transform(X_train)

# Create and fit the Random Forest Classifier
rf = RandomForestClassifier()
rf.fit(X_train_imputed, y_train)

X_test_imputed = imputer.fit_transform(X_test)

# Make predictions on the training set
y_pred = rf.predict(X_test_imputed)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

